# 05 Experiments

This notebook compares candidate algorithms and writes the best configuration back into `params.yaml`.

Current candidates:
- ridge
- random_forest
- gradient_boosting
- extra_trees

Optimization target:
- `test_rmse`


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "params.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

def read_json(path: str):
    return json.loads(Path(path).read_text(encoding="utf-8"))

params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))


In [2]:
params["experiments"]


{'models': ['ridge', 'random_forest', 'gradient_boosting', 'extra_trees'],
 'n_iter_per_model': 10,
 'cv_splits': 3,
 'n_jobs': -1,
 'random_state': 42,
 'optimization_metric': 'test_rmse',
 'log_to_mlflow': True,
 'mlflow_experiment_name': 'IronOre_Experiments',
 'results_path': 'dvc_pipeline/reports/experiment_results.csv',
 'best_result_path': 'dvc_pipeline/reports/best_experiment.json'}

In [3]:
subprocess.run([sys.executable, "dvc_pipeline/src/run_experiments.py"], check=True)
subprocess.run([sys.executable, "dvc_pipeline/src/update_params_from_experiments.py"], check=True)


CompletedProcess(args=['c:\\aditi\\.venv\\Scripts\\python.exe', 'dvc_pipeline/src/update_params_from_experiments.py'], returncode=0)

In [4]:
results_df = pd.read_csv(params["experiments"]["results_path"])
best_result = read_json(params["experiments"]["best_result_path"])
updated_params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))
results_df.sort_values("test_rmse"), best_result, updated_params["training"]


(          model_name  best_cv_rmse    train_rmse     train_mae    train_mape  \
 0              ridge      0.281728  1.719013e-01  5.270738e-02  6.205800e-04   
 1      random_forest     15.769963  1.460606e-01  3.990964e-02  4.172910e-04   
 2  gradient_boosting     15.546789  1.054079e-01  5.890439e-02  6.875990e-04   
 3        extra_trees     15.616871  7.652588e-13  6.317126e-13  6.312607e-15   
 
    train_r2  test_rmse  test_mae  test_mape   test_r2  \
 0  0.999978   0.029966  0.015143   0.000133  0.999994   
 1  0.999984   0.093777  0.053632   0.000493  0.999939   
 2  0.999992   0.099131  0.069094   0.000620  0.999932   
 3  1.000000   0.133031  0.039159   0.000360  0.999877   
 
                                          best_params  
 0               {"model__alpha": 0.2196385372416547}  
 1  {"model__n_estimators": 500, "model__min_sampl...  
 2  {"model__subsample": 1.0, "model__n_estimators...  
 3  {"model__n_estimators": 500, "model__min_sampl...  ,
 {'optimization_metr